# Batch FASTQ reader

Scan the selected FASTQ folder (no subfolders), generate a length-sorted CSV and histogram for every file, and align to a matching FASTA or SnapGene `.dna` reference when available.

Match references by the complete leading number: `1132.fastq.gz` matches `1132 LPPC EE R-S.dna`. Multiple matching references produce QC only and an ambiguity in the batch report. Missing references also produce QC only.

All outputs go directly into one separate output folder. Completed samples are skipped; set `OVERWRITE = True` to rebuild them. Failed or changed samples with existing outputs also require overwrite. The original input files are never modified.

Install Python dependencies with `python -m pip install -r requirements-notebook.txt`, and install `minimap2` and `samtools` separately for alignment. Open this notebook with its working directory set to the repository folder. In the prepared cloud environment, first run `source /workspace/fastq-reader-local/activate.sh`.

Sequencing-run prefixes are supported: `9M53XY_1_1108-G1.fastq` has sample ID **1108** and automatically matches a reference beginning with `1108`. The run token and barcode are ignored for matching; full FASTQ filenames remain the output/track labels.


In [ ]:
from pathlib import Path

FASTQ_DIR = Path("/path/to/fastq_folder")
REFERENCE_DIR = None  # None searches FASTQ_DIR; otherwise Path("/path/to/reference_folder")
REFERENCE_FILE = None  # leave None for automatic grouping by FASTQ ID
OUTPUT_DIR = FASTQ_DIR / "results"

OVERWRITE = False
QC_ONLY = False  # True skips reference search and alignment
BIN_SIZE_BP = 2000
LOG_Y = False
THREADS = 4


In [ ]:
import importlib
import batch_fastq_reader

# Reload after code updates so the kernel uses the current matcher.
importlib.invalidate_caches()
batch_fastq_reader = importlib.reload(batch_fastq_reader)
print("Using batch code:", batch_fastq_reader.__file__)
run_batch = batch_fastq_reader.run_batch

report = run_batch(
    fastq_dir=FASTQ_DIR,
    reference_dir=REFERENCE_DIR,
    reference_file=REFERENCE_FILE,
    output_dir=OUTPUT_DIR,
    overwrite=OVERWRITE,
    qc_only=QC_ONLY,
    bin_size=BIN_SIZE_BP,
    log_y=LOG_Y,
    threads=THREADS,
)
display(report)


Review the `status` and `message` columns for missing references, ambiguous matches, or failures. QC outputs are kept if alignment fails; the other samples continue processing. `batch_report.csv` contains the latest run's report.

Each FASTQ is processed separately, even if several filenames have the same sample number. Outputs normally use the FASTQ stem (`1132_summary.csv`, `1132_length_hist.png`, `1132_sorted.bam`). When multiple FASTQs share an identical stem, their full filenames become their output prefixes to prevent collisions.


## Open results in IGV

Each successful alignment now generates a saved session such as `1132_igv.xml`. In IGV, choose **File → Open Session** and select that XML file. It loads the matching reference FASTA and BAM together. Keep the XML, reference, FASTA index, BAM and BAM index in the same results folder.

To add sessions to existing results without rerunning the analysis, run this in a separate code cell:

```python
from batch_fastq_reader import create_igv_sessions
igv_report = create_igv_sessions(OUTPUT_DIR)
display(igv_report)
```

Existing sessions are preserved. Pass `overwrite=True` only if you want to replace them.

### One reference, multiple FASTQs

Automatic grouping is the default: `1183_2-3.fastq.gz`, `1183_4-6.fastq.gz`, `1183_12-17.fastq.gz`, and `1183_19-21.fastq.gz` all match one `1183` FASTA or SnapGene reference in the selected reference folder. Leave `REFERENCE_FILE = None`. They produce one combined session containing all four BAM tracks, labeled by the full FASTQ stem.

Only if you want to force FASTQs with different IDs to use one reference, set `REFERENCE_FILE = Path("/path/to/plasmid.dna")` (FASTA also works). This reference is used for every FASTQ, overriding ID-based folder matching. The results include individual sessions and one `<reference_name>_all_reads_igv.xml` session containing all compatible BAMs as separately labeled tracks. Open that combined XML in IGV. Combined sessions are refreshed on reruns; save customized overview sessions under a different filename.
